# Reconstruction Example Using lftomo

Environment Requirements:

See x

In [ ]:
from multiprocessing import shared_memory, Pool
from multiprocessing.managers import SharedMemoryManager
import os
from pathlib import Path
import sys

# Image Display
import cv2
from ipywidgets import Image
# Correct ordering if input not zero-padded
from natsort import natsorted
# Used for everything
import numpy as np
# Used to get processor data for setting multiprocessing.
import psutil
# Default file format is tif
import tifffile as tf
# Used for Recon
import tomopy

sys.path.insert(0, os.path.abspath('..'))

# Flexible logging to screen and file, in a parseable fixed-width format.
from lftomo.util.log import log
# Windowing to select only parts of an image at a time.
from lftomo.util.window import DataWindowSet, ProjOrder, ReconOrder, SinoOrder, get_tilt_pad, CTFov
# Handling flatfield backgrounds.
from lftomo.prep.flat import FLAT
# Shared Memory Handling
from lftomo.util.mem import SharedNP
# Reading and Writing files distributed
from lftomo.util.io import distribute_read, distribute_write, np_convert


## Setup

### Reconstruction Parameters

Starting with basics (input path, window size, output location)

In [ ]:
log.set_logdir(Path("..", "data", "logs"))
log.header()
# smm = SharedMemoryManager()
# smm.start()

In [ ]:

window_size = psutil.cpu_count()
input_path = Path("..", "data", "input", "Sam19")
flat_path = Path("..", "data", "input", "Sam19_Flats")
output_base = Path("..", "data", "output")

# Field of View Angles.
fov = CTFov(Theta=None, Degrees=180, Count=1)

internal_dtype = np.float32		# Data type to use internally. 
cf_count = 8 					# Number of centerfind sections to generate.
proc_count = 12					# Number of simulataneous processes.

In [ ]:
data_set = DataWindowSet.from_folder(input_path, ProjOrder, window_size, fov)
input_files = natsorted(Path(input_path).glob("**/*.tif*"))

### Flat Field Generation

In [ ]:
flat_mem = SharedNP(f'flats_ipynb', internal_dtype, data_set.data_shape._replace(Theta=len(FLAT))).create()

with flat_mem as flat_set:
	for flat in list(FLAT):
		flat_set[flat.index, :, :] = tf.imread(flat_path.joinpath(f"{flat}_median.tiff")).astype(internal_dtype)
	
	# Precalculation of pregain and postgain with darks.
	flat_set[FLAT.PREGAIN.index] -= flat_set[FLAT.PREDARK.index]
	flat_set[FLAT.POSTGAIN.index] -= flat_set[FLAT.POSTDARK.index]


In [ ]:
with flat_mem as flat_set:

	tf.imwrite("pregain.tiff", flat_set[FLAT.PREGAIN.index])
	tf.imwrite("postgain.tiff", flat_set[FLAT.POSTGAIN.index])
	tf.imwrite("predark.tiff", flat_set[FLAT.PREDARK.index])
	tf.imwrite("postdark.tiff", flat_set[FLAT.POSTDARK.index])

	if False:
		# Encode into in-memory PNG for Jupyter
		# _, pregain = cv2.imencode('.png', flat_set[FLAT.PREGAIN.index])
		# _, postgain = cv2.imencode('.png', flat_set[FLAT.POSTGAIN.index])

		# This 'works' but requires leveling
		Image(value=pregain, width=data_set.data_shape.X, height=data_set.data_shape.Y)

### Angle Calculation

In [ ]:
# No angle dropping so skipping for now.

## Center Finding

Steps:

1. Read Slice(s)
2. Data Cleanup
3. Center Calculation
4. Angle Calculation
5. Reconstruct Slices
6. Normalize Slices


In [ ]:
cf_mem = SharedNP(f"cf_ipynp", internal_dtype, data_set.data_shape._replace(Y=cf_count)).create()
cf_read_mem = SharedNP(f"cf_read_ipynb", data_set.dtype, SinoOrder(**data_set.data_shape._asdict())._replace(Y=cf_count)).create()

cf_dataset = data_set.at_interval(cf_count)

for i, cf_data in enumerate(cf_dataset):
	log.write("Center Finding", f"{cf_data}, {cf_read_mem.buffer_address}")
	distribute_read(cf_read_mem[i], cf_data, input_files)




In [ ]:
# with Pool(proc_count) as pool:
# 	output_base.joinpath("read_test").mkdir(exist_ok=True, parents=True)
# 	log.write("Writing CF Input")
# 	
# 	pool.starmap(distribute_write, [(cf_read_mem, output_base.joinpath("read_test", f"{str(i).zfill(5)}.tif"), i, "Theta")
# 									for i in range(cf_read_mem.shape.Theta)])

In [ ]:
cf_mem.close()
cf_mem.unlink()
cf_read_mem.close()
cf_read_mem.unlink()

## Reconstruction

Loop, looking at small windows each time.

1. Read Data
2. Clean Data (Pre-Rotation Steps)
3. Rotate Data
4. Clean Data (Post-Rotation Steps)
5. Reconstruct Slices
6. Normalize Slices

In [ ]:

with (SharedNP(f"sino_ipynb", internal_dtype, SinoOrder(**data_set.window_shape._asdict())) as sino_mem,
	SharedNP(f"work_ipynb", data_set.dtype, data_set.window_shape) as workmem_read,
	SharedNP(f"recon_ipynb", internal_dtype, ReconOrder(Y=data_set.window_size, Z=data_set.window_shape.X, X=data_set.window_shape.X)) as recon_mem):

	for j, data in enumerate(data_set):
		log.write("Reading Data", f"{data_set.window_shape} : {data}")
		distribute_read(workmem_read[0:data_set.window_shape.Theta], data, input_files)

# 		with Pool(proc_count) as pool:
# 			output_base.joinpath("fullread_test").mkdir(exist_ok=True, parents=True)
# 			pool.starmap(distribute_write, [(workmem_read, output_base.joinpath("fullread_test", f"{str(j).zfill(3)}_{str(i).zfill(5)}.tif"), i)
# 											for i in range(data_set.window_shape.Theta)])

		with workmem_read as input_data, recon_mem as output_data:
			output_data[:, :, :] = tomopy.recon(input_data[:, :, :], data_set.angles, center=1028, algorithm="gridrec", ncore=proc_count)	

		with Pool(proc_count) as pool:
			output_base.joinpath("recon").mkdir(exist_ok=True, parents=True)
			log.write("Writing Recon", f"{data.Y()}")
			
			pool.starmap(distribute_write, [(recon_mem[0:data_set.window_size], output_base.joinpath("recon", f"{str(i).zfill(5)}.tif"), i - data.Y().start)
											for i in data.Y()])


In [ ]:
flat_mem.close()
flat_mem.unlink()

In [ ]:
sino_mem.close()
sino_mem.unlink()
workmem_read.close()
workmem_read.unlink()
recon_mem.close()
recon_mem.unlink()